# Zipformer code-switch (VI+EN) fine-tuning on Colab

End-to-end on Colab + Google Drive:
**setup -> select VI+EN utterances -> materialize combined dataset -> compute fbank -> resumable fine-tune.**

Reuses `compute_fbank_huggingface.py` and `finetune.py` from this repo. Heavy
artifacts (cuts/feats + checkpoints) live on Drive so they survive runtime
restarts. You must place **your own** `bpe.model` and `pretrained.pt` on Drive.

Run cells top-to-bottom. Use a CPU runtime for selection/materialize/fbank and a
GPU runtime for fine-tuning.

## 0. Setup

In [1]:
import os

# --- EDIT THESE -----------------------------------------------------------
REPO_URL   = ''  # optional: git URL of THIS repo. Leave '' if you upload it.
REPO       = '.'                 # where this repo lives
DRIVE      = 'vibank_cs/'       # persistent workspace
PREFIX     = 'vibank_cs'
BPE_MODEL  = f'model/bpe.model'        # <- upload YOUR bpe.model here
PRETRAINED = f'model/epoch-35-avg-6.pt'    # <- upload YOUR checkpoint here
TEXT_NORM  = 'none'   # MUST match how your BPE was trained (see BPE check)
# -------------------------------------------------------------------------

ASR      = f'{REPO}/icefall/egs/librispeech/ASR'
FBANK    = f'data/fbank'
EXP_DIR  = f'data/exp_finetune'
COMBINED = f'data/combined_dataset'
MANIFEST = f'data/combined_manifest.jsonl'
os.makedirs(DRIVE, exist_ok=True)

if REPO_URL and not os.path.isdir(REPO):
    !git clone {REPO_URL} {REPO}
assert os.path.isdir(REPO), f'Put this repo at {REPO} (clone or upload).'
print('REPO  =', REPO)
print('DRIVE =', DRIVE)

# --- HF Hub persistence (survives Colab restarts; create repos PRIVATE) ---
HF_DATA_REPO  = ''   # e.g. 'you/vibank-cs-data'  -> combined_dataset + fbank cuts
HF_MODEL_REPO = ''   # e.g. 'you/vibank-cs-model' -> bpe.model, base + epoch-N.pt
HF_SYNC       = os.path.abspath(f'{REPO}/colab/hf_sync.py')  # absolute: survives %cd


REPO  = .
DRIVE = vibank_cs/


In [ ]:
# Python deps. k2 is the fragile one: it must match Colab's torch + CUDA.
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)

!pip -q install lhotse datasets soundfile sentencepiece wordfreq kaldialign num2words huggingface_hub mlflow
# k2: pick the wheel matching the torch/cuda printed above. The -f index lists
# all builds; if the bare install fails, copy the exact matching URL from there.
!pip -q install k2 -f https://k2-fsa.github.io/k2/cuda.html || \
  echo 'k2 install failed - open https://k2-fsa.github.io/k2/cuda.html and pick the wheel for the torch/cuda above'
!pip -q install -e icefall

import k2, lhotse, datasets
print('k2:', k2.__version__, '| lhotse:', lhotse.__version__, '| datasets:', datasets.__version__)

In [ ]:
# Place the repo's maintained scripts where the recipe expects them.
import shutil
shutil.copy(f'{REPO}/compute_fbank_huggingface.py', f'{ASR}/local/compute_fbank_huggingface.py')
shutil.copy(f'{REPO}/finetune.py', f'{ASR}/zipformer/finetune.py')
shutil.copy(f'{REPO}/decode.py', f'{ASR}/zipformer/decode.py')
shutil.copy(f'{REPO}/ctc_decode.py', f'{ASR}/zipformer/ctc_decode.py')
print('scripts placed under', ASR)

## 0b. Hugging Face Hub (persistence)

Log in once per session, then push/restore artifacts. Leave the
`HF_*_REPO` vars empty to skip the Hub and use local paths only.


In [ ]:
# Auth once per session (or set os.environ['HF_TOKEN'] = '...').
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# Inputs (bpe.model + base ckpt): push them once, or restore after a restart.
if HF_MODEL_REPO:
    if os.path.exists(PRETRAINED):
        !python {HF_SYNC} push --repo-id {HF_MODEL_REPO} --repo-type model --private --local {BPE_MODEL}
        !python {HF_SYNC} push --repo-id {HF_MODEL_REPO} --repo-type model --private --local {PRETRAINED}
    else:
        !python {HF_SYNC} pull --repo-id {HF_MODEL_REPO} --repo-type model --local model \
          --allow-pattern 'bpe.model' --allow-pattern 'epoch-35-avg-*.pt'
else:
    print('HF_MODEL_REPO empty - skipping input push/restore.')


## 1. Filter sanity check

In [ ]:
%cd {REPO}/colab
!python test_codeswitch_filter.py

## 2. Select VI+EN utterances

Edit `sources` with your candidate datasets. `text_key`/`audio_key` are the
column names in each dataset. Start with `--limit` for a dry run, review the
samples, tune, then drop `--limit` for the full scan.

In [ ]:
import json
sources = [
    {'dataset': 'org/your-vi-dataset-1', 'config': None, 'split': 'train',
     'text_key': 'transcription', 'audio_key': 'audio'},
    # {'dataset': 'org/your-vi-dataset-2', 'split': 'train', 'text_key': 'sentence'},
]
with open('sources.json', 'w', encoding='utf-8') as f:
    json.dump(sources, f, ensure_ascii=False, indent=2)
print(open('sources.json', encoding='utf-8').read())

In [ ]:
# Dry run: scan up to 5000 rows/source. Remove --limit for the real selection.
!python select_codeswitch_utterances.py \
  --sources-json sources.json \
  --output-dir {DRIVE} \
  --min-en-tokens 1 \
  --limit 5000

In [ ]:
# Review precision before scaling. Tune --min-en-tokens / --max-en-ratio /
# wordlists if too many false positives appear here.
print('=== SELECTED (english tokens shown) ===')
print(open(f'{DRIVE}/samples_selected.txt', encoding='utf-8').read()[:4000])
print('\n=== REJECTED ===')
print(open(f'{DRIVE}/samples_rejected.txt', encoding='utf-8').read()[:2000])
print('\n=== STATS ===')
print(open(f'{DRIVE}/selection_stats.json', encoding='utf-8').read())

## 3. BPE coverage check (BLOCKING)

Confirm your BPE tokenizes the selected (normalized) transcripts without an
`<unk>` flood and that `TEXT_NORM` round-trips Vietnamese correctly. If this
looks wrong, fix `TEXT_NORM` (likely `'none'`) before computing any features.

In [ ]:
import json, sentencepiece as spm, sys
sys.path.insert(0, f'{ASR}/local')
from compute_fbank_huggingface import normalize_text

sp = spm.SentencePieceProcessor(); sp.load(BPE_MODEL)
unk = sp.piece_to_id('<unk>')
rows = [json.loads(l) for l in open(MANIFEST, encoding='utf-8')][:20]
tot = unkn = 0
for r in rows:
    norm = normalize_text(r['text'], TEXT_NORM)
    ids = sp.encode(norm, out_type=int)
    tot += len(ids); unkn += sum(i == unk for i in ids)
    print(f"{norm}\n  -> {sp.encode(norm, out_type=str)}\n")
print(f'tokens={tot} unk={unkn} ({100*unkn/max(tot,1):.1f}%)')
assert tot == 0 or unkn / tot < 0.02, 'Too many <unk>; check TEXT_NORM / BPE.'

## 4. Materialize combined dataset to Drive

In [ ]:
!python materialize_dataset.py \
  --manifest {MANIFEST} \
  --output-dir {COMBINED} \
  --sampling-rate 16000 \
  --dev-frac 0.02 \
  --test-frac 0.02

## 5. Compute fbank (train perturbed, dev clean) -> Drive

`--load-from-disk` reads the combined dataset; `--split train|dev` selects the
DatasetDict split saved above.

In [ ]:
%cd {ASR}
import os
# After a restart: pull dataset + fbank from the Hub to skip recompute.
need_fbank = not os.path.exists(f'{FBANK}/{PREFIX}_cuts_train.jsonl.gz')
if HF_DATA_REPO and need_fbank:
    !python {HF_SYNC} pull --repo-id {HF_DATA_REPO} --repo-type dataset --local data --path-in-repo combined_dataset
    !python {HF_SYNC} pull --repo-id {HF_DATA_REPO} --repo-type dataset --local data --path-in-repo fbank
if os.path.exists(f'{FBANK}/{PREFIX}_cuts_train.jsonl.gz'):
    print('Fbank cuts present - you may SKIP the compute cell below.')
else:
    print('No fbank cuts present - RUN the compute cell below.')


In [ ]:
%cd {ASR}
import os, json
os.makedirs(FBANK, exist_ok=True)

# Compute fbank for whichever splits materialize_dataset.py actually produced.
# 'train' gets 3x speed perturbation; 'dev'/'test' stay clean. If 'test' is
# missing here, re-run the materialize cell with --test-frac > 0 to create it,
# then re-run this cell (the held-out test set powers the before/after WER).
with open(f'{COMBINED}/dataset_dict.json') as f:
    splits = json.load(f)['splits']
print('combined_dataset splits:', splits)

for split in splits:
    perturb = '--perturb-speed' if split == 'train' else ''
    !python local/compute_fbank_huggingface.py \
      --load-from-disk --dataset {COMBINED} --split {split} \
      --output-name {split} --prefix {PREFIX} --output-dir {FBANK} \
      --text-normalization {TEXT_NORM} --resample-rate 16000 \
      --bpe-model {BPE_MODEL} {perturb}

In [ ]:
# Persist dataset + fbank cuts to the Hub (run after fbank computes).
if HF_DATA_REPO:
    !python {HF_SYNC} push --repo-id {HF_DATA_REPO} --repo-type dataset --private --local {COMBINED} --path-in-repo combined_dataset
    # whole fbank dir: cuts (*.jsonl.gz) AND the lilcom *_feats_* features they reference.
    !python {HF_SYNC} push --repo-id {HF_DATA_REPO} --repo-type dataset --private --local {FBANK} --path-in-repo fbank
else:
    print('HF_DATA_REPO empty - skipping dataset/fbank push.')


In [ ]:
# Duration distribution sanity check.
!python local/display_manifest_statistics.py 2>/dev/null || \
  python -c "from lhotse import load_manifest_lazy; \
cs=load_manifest_lazy('{FBANK}/{PREFIX}_cuts_train.jsonl.gz'); \
import itertools; print('train cuts:', len(list(itertools.islice((c for c in cs),0,10**9))))"

## 5b. MUSAN noise prep (CPU)

Download MUSAN, compute its fbank (80 mel bins / 16 kHz, matching train), and
build a **speech-only** subset (human background). Then build a **noisy copy**
of the held-out test set to measure noise robustness. All artifacts land in
`{FBANK}`, so the existing Hub push/pull cells persist them automatically.

In [ ]:
%cd {ASR}
import os
# MUSAN fbank (matches train: 80 mel bins, 16 kHz). Skips if already present
# (e.g. pulled from the Hub with the rest of FBANK). Raw corpus ~11 GB.
MUSAN_DL = 'data/musan_dl'  # raw download dir (not persisted)
if not os.path.exists(f'{FBANK}/musan_cuts.jsonl.gz'):
    if not os.path.isdir(f'{MUSAN_DL}/musan'):
        !lhotse download musan {MUSAN_DL}
    !lhotse prepare musan {MUSAN_DL}/musan data/manifests
    !python local/compute_fbank_musan.py --output-dir {FBANK}
else:
    print('musan_cuts.jsonl.gz present - skipping MUSAN compute.')

In [ ]:
%cd {ASR}
# Speech-only MUSAN manifest (human background) for MUSAN_SOURCE='speech'.
from lhotse import load_manifest
musan = load_manifest(f'{FBANK}/musan_cuts.jsonl.gz')
speech = musan.filter(lambda c: (c.recording_id or '').startswith('speech')).to_eager()
out = f'{FBANK}/musan_speech_cuts.jsonl.gz'
speech.to_file(out)
print(f'speech cuts: {len(speech)} -> {out}')

In [ ]:
%cd {ASR}
# Build a noisy copy of the held-out test set (reuses the training CutMix).
MK_NOISY = HF_SYNC.replace('hf_sync.py', 'make_noisy_cuts.py')
!python {MK_NOISY}   --test-cuts {FBANK}/{PREFIX}_cuts_test.jsonl.gz   --musan-cuts {FBANK}/musan_cuts.jsonl.gz   --snr-low 5 --snr-high 20   --output {FBANK}/{PREFIX}_cuts_test_noisy.jsonl.gz

In [ ]:
# Persist MUSAN + noisy artifacts to the Hub so future sessions skip the
# ~11 GB download AND the fbank compute. Pushes the whole fbank dir to the same
# 'fbank' subtree; upload_folder dedups the already-pushed train/dev/test feats,
# so effectively only musan_cuts / musan_speech_cuts / musan_feats* /
# {PREFIX}_cuts_test_noisy are uploaded. Restored by the section-5 pull cell.
if HF_DATA_REPO:
    !python {HF_SYNC} push --repo-id {HF_DATA_REPO} --repo-type dataset --private \
      --local {FBANK} --path-in-repo fbank
else:
    print('HF_DATA_REPO empty - skipping MUSAN/noisy push.')

## 6. Fine-tune (GPU, resumable)

Switch the runtime to **GPU** first. Checkpoints write to `EXP_DIR` locally.
Run the **Resume** cell to pull the latest `epoch-N.pt` from the Hub and set
`START_EPOCH`/`DO_FINETUNE` automatically, then push checkpoints on demand.


In [ ]:
# MLflow on Databricks (optional): metrics + best weight pushed to your workspace.
# Leave USE_MLFLOW = 0 to skip entirely. Token is read via getpass - never hardcode it.
import os, getpass

USE_MLFLOW = 1  # set 0 to disable MLflow logging

if USE_MLFLOW:
    os.environ['MLFLOW_TRACKING_URI'] = 'databricks'
    os.environ['DATABRICKS_HOST'] = 'https://<your-workspace>.cloud.databricks.com'  # <- EDIT
    if not os.environ.get('DATABRICKS_TOKEN'):
        os.environ['DATABRICKS_TOKEN'] = getpass.getpass('Databricks token: ')
    MLFLOW_EXPERIMENT = '/Users/taiinguyenn139@gmail.com/zipformer-cs-finetune'  # <- EDIT
    MLFLOW_RUN_NAME   = PREFIX
    MLFLOW_ARGS = (f'--use-mlflow 1 --mlflow-experiment {MLFLOW_EXPERIMENT} '
                   f'--mlflow-run-name {MLFLOW_RUN_NAME}')
else:
    MLFLOW_ARGS = '--use-mlflow 0'
print('MLFLOW_ARGS =', MLFLOW_ARGS)

In [ ]:
# MUSAN noise augmentation config. ENABLE_MUSAN=0 to skip.
# MUSAN_SOURCE: 'full' (music+noise+speech) or 'speech' (human background only).
ENABLE_MUSAN = 1
MUSAN_SOURCE = 'full'      # 'full' | 'speech'
SNR_LOW, SNR_HIGH = 5, 20  # lower = harsher background

_musan_cuts = {'full': 'musan_cuts.jsonl.gz',
               'speech': 'musan_speech_cuts.jsonl.gz'}[MUSAN_SOURCE]
if ENABLE_MUSAN:
    MUSAN_ARGS = (f'--enable-musan 1 --musan-cuts {_musan_cuts} '
                  f'--musan-snr-low {SNR_LOW} --musan-snr-high {SNR_HIGH}')
else:
    MUSAN_ARGS = '--enable-musan 0'
print('MUSAN_ARGS =', MUSAN_ARGS)

In [ ]:
# CTC head config. Set USE_CTC=1 if you fine-tune / evaluate the CTC head
# (joint CTC+transducer). Threads --use-ctc through fine-tune and the WER decode
# so the model is built with the matching architecture.
USE_CTC = 1
CTC_ARGS = f'--use-ctc {USE_CTC}'
print('CTC_ARGS =', CTC_ARGS)

In [ ]:
# Smoke test: 1 epoch, tiny batch. Confirms ckpt loads + a checkpoint is written.
%cd {ASR}
!python zipformer/finetune.py \
  --world-size 1 --num-epochs 1 --start-epoch 1 --use-fp16 1 \
  --do-finetune 1 --finetune-ckpt {PRETRAINED} --base-lr 0.0045 --use-mux 0 \
  --bpe-model {BPE_MODEL} --exp-dir {EXP_DIR} \
  --manifest-dir {FBANK} \
  --train-manifest {PREFIX}_cuts_train.jsonl.gz \
  --valid-manifest {PREFIX}_cuts_dev.jsonl.gz \
  --valid-set-name {PREFIX} \
  {MLFLOW_ARGS} \
  {CTC_ARGS} \
  {MUSAN_ARGS} --max-duration 80

In [ ]:
%cd {ASR}
# Resume: pull the latest epoch-N.pt from the Hub, derive --start-epoch.
START_EPOCH, DO_FINETUNE = 1, 1
if HF_MODEL_REPO:
    out = !python {HF_SYNC} pull-latest-ckpt --repo-id {HF_MODEL_REPO} --local {EXP_DIR}
    last = next((l.strip() for l in reversed(out) if l.strip().isdigit()), '')
    if last:
        START_EPOCH, DO_FINETUNE = int(last) + 1, 0
        print(f'Resuming from epoch {last} -> --start-epoch {START_EPOCH}')
    else:
        print('No checkpoint on the Hub - starting fresh.')
print('START_EPOCH =', START_EPOCH, '| DO_FINETUNE =', DO_FINETUNE)

In [ ]:
# Live training monitor. finetune.py logs to {EXP_DIR}/tensorboard by default.
# Launch this BEFORE the full run below; it refreshes as training writes scalars
# (train/tot_loss, train/{PREFIX}_valid_loss, learning rate, grad scale).
%cd {ASR}
%load_ext tensorboard
%tensorboard --logdir {EXP_DIR}/tensorboard

In [ ]:
# Full run. START_EPOCH / DO_FINETUNE come from the Resume cell above
# (fresh: 1 / 1; resuming: latest_epoch+1 / 0).
%cd {ASR}
!python zipformer/finetune.py \
  --world-size 1 --num-epochs 20 --start-epoch {START_EPOCH} --use-fp16 1 \
  --do-finetune {DO_FINETUNE} --finetune-ckpt {PRETRAINED} --base-lr 0.0045 --use-mux 0 \
  --bpe-model {BPE_MODEL} --exp-dir {EXP_DIR} \
  --manifest-dir {FBANK} \
  --train-manifest {PREFIX}_cuts_train.jsonl.gz \
  --valid-manifest {PREFIX}_cuts_dev.jsonl.gz \
  --valid-set-name {PREFIX} \
  {MLFLOW_ARGS} \
  {CTC_ARGS} \
  {MUSAN_ARGS} --max-duration 300

In [ ]:
# On-demand: push checkpoints written so far to the model repo.
# Re-run anytime (e.g. before the session times out) to back up progress.
if HF_MODEL_REPO:
    !python {HF_SYNC} push --repo-id {HF_MODEL_REPO} --repo-type model --private \
      --local {EXP_DIR} --allow-pattern 'epoch-*.pt'
else:
    print('HF_MODEL_REPO empty - skipping checkpoint push.')

## 7. Before/after WER (GPU)

Decode the **base** checkpoint and the **fine-tuned** weight(s) on the held-out
`test` split with greedy search, then compare WER. Same cuts, same BPE, so the
numbers are directly comparable. A base WER near 100% means a BPE/`TEXT_NORM`
mismatch, not a real baseline - fix that before trusting any number.

In [ ]:
# Before/after WER on the held-out test split(s).
# Transducer head -> decode.py greedy_search; CTC head -> ctc_decode.py
# ctc-greedy-search (only when USE_CTC). Both use the same cuts so the columns
# are directly comparable. SWEEP_ALL=False -> latest epoch; True -> every epoch.
%cd {ASR}
import glob, re, os

SWEEP_ALL = False

def run_transducer(checkpoint, out_dir, test_manifest):
    os.makedirs(out_dir, exist_ok=True)
    !python zipformer/decode.py \
      --checkpoint {checkpoint} --decoding-method greedy_search --use-ctc {USE_CTC} \
      --bpe-model {BPE_MODEL} --exp-dir {out_dir} \
      --manifest-dir {FBANK} --test-manifest {test_manifest} \
      --test-set-name {PREFIX} --max-duration 300

def run_ctc(checkpoint, out_dir, test_manifest):
    os.makedirs(out_dir, exist_ok=True)
    !python zipformer/ctc_decode.py \
      --checkpoint {checkpoint} --decoding-method ctc-greedy-search --use-ctc 1 \
      --bpe-model {BPE_MODEL} --exp-dir {out_dir} \
      --manifest-dir {FBANK} --test-manifest {test_manifest} \
      --test-set-name {PREFIX} --max-duration 300

def read_wer(out_dir, subdir):
    wers = []
    for fp in glob.glob(f'{out_dir}/{subdir}/wer-summary-{PREFIX}-*.txt'):
        for line in open(fp, encoding='utf8').read().splitlines()[1:]:  # skip header
            parts = line.split('\t')
            if len(parts) == 2:
                try:
                    wers.append(float(parts[1]))
                except ValueError:
                    pass
    return min(wers) if wers else None

# Test conditions: clean always; noisy if section 5b generated it.
conditions = [('clean', f'{PREFIX}_cuts_test.jsonl.gz')]
if os.path.exists(f'{FBANK}/{PREFIX}_cuts_test_noisy.jsonl.gz'):
    conditions.append(('noisy', f'{PREFIX}_cuts_test_noisy.jsonl.gz'))

# Checkpoints: base + latest (or all) fine-tuned epochs.
epochs = sorted(int(re.search(r'epoch-(\d+)\.pt', p).group(1))
                for p in glob.glob(f'{EXP_DIR}/epoch-*.pt'))
ckpts = [('base', PRETRAINED)]
for ep in (epochs if SWEEP_ALL else epochs[-1:]):
    ckpts.append((f'epoch-{ep}', f'{EXP_DIR}/epoch-{ep}.pt'))

# Heads: transducer always; ctc when USE_CTC. (runner, result subdir)
heads = [('transducer', run_transducer, 'greedy_search')]
if int(USE_CTC):
    heads.append(('ctc', run_ctc, 'ctc-greedy-search'))

results = {}
for cond, manifest in conditions:
    for label, ckpt in ckpts:
        for hname, runner, subdir in heads:
            out_dir = f'data/exp_wer_{label}_{cond}_{hname}'
            runner(ckpt, out_dir, manifest)
            results[(label, cond, hname)] = read_wer(out_dir, subdir)

cols = [(c, h) for c, _ in conditions for h, _, _ in heads]
print('\n=== WER% (greedy) on', PREFIX, 'test ===')
print(f'{"checkpoint":<14}' + ''.join(f'{c+"/"+h:>16}' for c, h in cols))
for label, _ in ckpts:
    row = ''.join(
        f'{("n/a" if results[(label,c,h)] is None else f"{results[(label,c,h)]:.2f}"):>16}'
        for c, h in cols)
    print(f'{label:<14}{row}')